# Prática 1 — Feature Extraction em CIFAR-10 (Versão Professor)
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook vamos demonstrar o conceito de **Feature Extraction** utilizando uma rede convolucional profunda pré-treinada (**ResNet-18**):
1. Congelar todos os parâmetros convolucionais do modelo (backbone).
2. Substituir a camada final (totalmente conectada) por uma nova linear adaptada ao CIFAR-10.
3. Treinar apenas a nova camada usando SGD.
4. Avaliar a performance do modelo.

**Referência:** Wang & Chen (2023), *Introduction to Transfer Learning: Algorithms and Practice*, Springer.

## 0. Imports e Configuração de Hardware

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

# Configurar dispositivo (GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Subset Balanceado

> **IMPORTANTE (Didático):** O dataset completo do CIFAR-10 tem 50.000 imagens de treino. Treinar nele em CPUs comuns de laboratório levaria muito tempo (> 2 horas). Por isso, implementamos uma função que extrai um **subset balanceado com 500 imagens por classe** (total de 5.000 imagens para treino e 1.000 para validação).

In [ ]:
# Hiperparâmetros de amostragem
N_TRAIN = 5000   # 500 amostras por classe (10% do total)
N_VAL = 1000     # 100 amostras por classe

# Pipelines de transformação (ResNet-18 exige 224x224 e normalização do ImageNet)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

# Função para extrair um subset com classes balanceadas
def extract_balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, N_TRAIN)
val_dataset = extract_balanced_subset(val_full, N_VAL)

# DataLoaders (num_workers=0 garante estabilidade absoluta no Windows)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Amostras de Treino: {len(train_dataset)} | Validação: {len(val_dataset)}')

## 2. Preparação do Modelo (Feature Extraction)

Aqui, carregamos a **ResNet-18** com pesos pré-treinados do ImageNet, congelamos todo o extrator de features convolucional e criamos um novo classificador linear final.

In [ ]:
# Carregar pesos pré-treinados do ImageNet
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# PASSO CRUCIAL: Congelar todos os gradientes do backbone (filtros convolucionais)
for param in model.parameters():
    param.requires_grad = False

# Substituir a camada classificadora fc final por uma adaptada ao CIFAR-10
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)  # CIFAR-10 tem 10 classes

model = model.to(device)

# Contar parâmetros treináveis vs. congelados
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (apenas a última camada)')

## 3. Função de Treinamento e Avaliação

In [ ]:
criterion = nn.CrossEntropyLoss()
# Otimizador configurado para treinar APENAS os parâmetros de model.fc
optimizer = optim.SGD(model.fc.parameters(), lr=0.01, momentum=0.9)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

## 4. Execução do Experimento

Vamos rodar por apenas 5 épocas.

In [ ]:
num_epochs = 5
print('=== Iniciando Treinamento (Feature Extraction) ===')
t_start = time.time()

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    loss = train_epoch(model, train_loader, criterion, optimizer, device)
    acc = evaluate(model, val_loader, device)
    elapsed = time.time() - t0
    print(f'Época {epoch}/{num_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {elapsed:.1f}s')

total_time = time.time() - t_start
print(f'Treinamento completo em {total_time/60:.1f} minutos.')

## 5. Questões Teóricas e de Reflexão (Didático)

1. **Por que a acurácia de validação já inicia tão alta na época 1 (geralmente > 70%)?**  
   *Resposta:* A ResNet-18 foi pré-treinada no ImageNet, aprendendo filtros robustos para detectar bordas, texturas, formas e objetos. Mesmo mudando o conjunto de dados para o CIFAR-10, essas representações genéricas continuam extremamente úteis e discriminativas.

2. **Qual é o efeito prático do congelamento em relação ao tempo de treinamento em CPU?**  
   *Resposta:* Ao congelar o backbone, não precisamos computar gradientes para os 11 milhões de parâmetros convolucionais na retropropagação (backpropagation). Apenas os gradientes da camada de classificação linear (5.130 parâmetros) são calculados, reduzindo drasticamente o esforço de processamento por batch.